In [65]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor,AdaBoostClassifier

In [66]:
# load the dataset
df = pd.read_csv('Ensemble_learning_datasets.csv')
df.head(2)

,Age,Annual_Income,Credit_Score,Employment_Years,Debt_to_Income_Ratio,Loan_Amount,Interest_Rate,Monthly_Expenses,Savings_Balance,Late_Payments_Last_Year,Existing_Loans_Count,Education_Level,Employment_Type,Home_Ownership,Marital_Status,Loan_Status
0,22.0,48671.773932,642.511200,2.729125,0.401255,10956.552138,10.698067,2324.860726,10324.808493,0.0,3.0,Bachelor,Salaried,Mortgage,Single,Approved
1,59.0,85335.870658,791.727009,8.117880,0.560087,23191.755595,10.241296,5044.756635,7053.268932,0.0,3.0,Bachelor,Salaried,Rent,Single,Rejected


In [67]:
# check the shape of the dataset
df.shape

(2000, 16)

In [68]:
# check for null values
df.isnull().sum()

Age                        17
Annual_Income              21
Credit_Score               24
Employment_Years           19
Debt_to_Income_Ratio       19
Loan_Amount                18
Interest_Rate              14
Monthly_Expenses           20
Savings_Balance            18
Late_Payments_Last_Year    21
Existing_Loans_Count       20
Education_Level            20
Employment_Type            19
Home_Ownership             28
Marital_Status             22
Loan_Status                 0
dtype: int64

In [69]:
# handle missing values
# divide numerical and categorical columns
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include='object').columns.tolist()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14276\2813690692.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include='object').columns.tolist()


In [70]:
# Fill missing values in numerical columns with the median
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())   

In [71]:
# fill missing value for categorical columns with mode

for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0]) 

In [72]:
# after handling missing values, check for null values again
df.isnull().sum()

Age                        0
Annual_Income              0
Credit_Score               0
Employment_Years           0
Debt_to_Income_Ratio       0
Loan_Amount                0
Interest_Rate              0
Monthly_Expenses           0
Savings_Balance            0
Late_Payments_Last_Year    0
Existing_Loans_Count       0
Education_Level            0
Employment_Type            0
Home_Ownership             0
Marital_Status             0
Loan_Status                0
dtype: int64

In [73]:
# check outliers in numerical columns
for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Outliers in {col}: {outliers.shape[0]}")

Outliers in Age: 2
Outliers in Annual_Income: 12
Outliers in Credit_Score: 27
Outliers in Employment_Years: 55
Outliers in Debt_to_Income_Ratio: 16
Outliers in Loan_Amount: 6
Outliers in Interest_Rate: 7
Outliers in Monthly_Expenses: 17
Outliers in Savings_Balance: 142
Outliers in Late_Payments_Last_Year: 65
Outliers in Existing_Loans_Count: 1


In [74]:
# handle outliers by clipping the values to the lower and upper bounds
for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound) 
    df[col]

In [75]:
# check duplicate rows
df.duplicated().sum()

np.int64(0)

In [76]:
# check categorical columns
for col in categorical_columns:
    print(f"Unique values in {col}: {df[col].unique()}")

Unique values in Education_Level: <StringArray>
['Bachelor', 'High School', 'Diploma', 'Master']
Length: 4, dtype: str
Unique values in Employment_Type: <StringArray>
['Salaried', 'Contract', 'Self-employed', 'Unemployed']
Length: 4, dtype: str
Unique values in Home_Ownership: <StringArray>
['Mortgage', 'Rent', 'Own']
Length: 3, dtype: str
Unique values in Marital_Status: <StringArray>
['Single', 'Married', 'Divorced']
Length: 3, dtype: str
Unique values in Loan_Status: <StringArray>
['Approved', 'Rejected']
Length: 2, dtype: str


In [77]:
# check Loan_Status column values
df['Loan_Status'].dtype
# replace categorical values with numerical values
df['Loan_Status'] = df['Loan_Status'].replace({'Approved': 1, 'Rejected': 0})
df['Loan_Status'].unique()

array([1, 0], dtype=object)

In [78]:
categorical_columns

['Education_Level',
 'Employment_Type',
 'Home_Ownership',
 'Marital_Status',
 'Loan_Status']

In [79]:
print(df.columns.tolist())

['Age', 'Annual_Income', 'Credit_Score', 'Employment_Years', 'Debt_to_Income_Ratio', 'Loan_Amount', 'Interest_Rate', 'Monthly_Expenses', 'Savings_Balance', 'Late_Payments_Last_Year', 'Existing_Loans_Count', 'Education_Level', 'Employment_Type', 'Home_Ownership', 'Marital_Status', 'Loan_Status']


In [80]:
# handel categorical columns by dummy encoding
import pandas as pd

categorical_columns = [
    "Education_Level",
    "Employment_Type",
    "Home_Ownership",
    "Marital_Status"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)

print(df.head())

    Age  Annual_Income  Credit_Score  Employment_Years  Debt_to_Income_Ratio  \
0  22.0   48671.773932    642.511200          2.729125              0.401255   
1  59.0   85335.870658    791.727009          8.117880              0.560087   
2  52.0   65761.461232    678.213912          2.042937              0.649237   
3  41.0   58778.493519    736.542946          0.356053              0.415917   
4  40.0   62663.997526    677.068342          4.127916              0.111530   

    Loan_Amount  Interest_Rate  Monthly_Expenses  Savings_Balance  \
0  10956.552138      10.698067       2324.860726     10324.808493   
1  23191.755595      10.241296       5044.756635      7053.268932   
2  28706.503206      14.419877       2024.946180      8347.090074   
3  12531.595536      16.745638       2694.854044     11017.058684   
4  11661.094566       9.205600       1680.228814     35919.573427   

   Late_Payments_Last_Year  ...  Employment_Type_Contract  \
0                      0.0  ...            

In [81]:
df.columns

Index(['Age', 'Annual_Income', 'Credit_Score', 'Employment_Years',
       'Debt_to_Income_Ratio', 'Loan_Amount', 'Interest_Rate',
       'Monthly_Expenses', 'Savings_Balance', 'Late_Payments_Last_Year',
       'Existing_Loans_Count', 'Loan_Status', 'Education_Level_Bachelor',
       'Education_Level_Diploma', 'Education_Level_High School',
       'Education_Level_Master', 'Employment_Type_Contract',
       'Employment_Type_Salaried', 'Employment_Type_Self-employed',
       'Employment_Type_Unemployed', 'Home_Ownership_Mortgage',
       'Home_Ownership_Own', 'Home_Ownership_Rent', 'Marital_Status_Divorced',
       'Marital_Status_Married', 'Marital_Status_Single'],
      dtype='str')

In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            2000 non-null   float64
 1   Annual_Income                  2000 non-null   float64
 2   Credit_Score                   2000 non-null   float64
 3   Employment_Years               2000 non-null   float64
 4   Debt_to_Income_Ratio           2000 non-null   float64
 5   Loan_Amount                    2000 non-null   float64
 6   Interest_Rate                  2000 non-null   float64
 7   Monthly_Expenses               2000 non-null   float64
 8   Savings_Balance                2000 non-null   float64
 9   Late_Payments_Last_Year        2000 non-null   float64
 10  Existing_Loans_Count           2000 non-null   float64
 11  Loan_Status                    2000 non-null   object 
 12  Education_Level_Bachelor       2000 non-null   int64  
 13 

In [83]:
# find categorical columns
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
categorical_columns

['Loan_Status']

In [84]:
# convert loan_status column to int
df['Loan_Status'] = df['Loan_Status'].astype(int)
df['Loan_Status']


0       1
1       0
2       0
3       1
4       0
       ..
1995    0
1996    1
1997    1
1998    0
1999    0
Name: Loan_Status, Length: 2000, dtype: int64

In [85]:
# divide the dataset into features and target variable
X=df.drop('Loan_Status', axis=1)
y=df['Loan_Status']

In [86]:
print(X.shape)
print(y.shape)

(2000, 25)
(2000,)


In [87]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [94]:
# create logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
model=LogisticRegression()
#train the model
model.fit(X_train,y_train)
# prect the model
y_pred=model.predict(X_test)
# check accuracy score
accuracy=accuracy_score(y_test,y_pred)
print('Accuracy:', accuracy*100)
print('Precision:', precision_score(y_test,y_pred))
print('Recall:', recall_score(y_test,y_pred))
print('F1 Score:', f1_score(y_test,y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test,y_pred))
print('Classification Report:')
print(classification_report(y_test,y_pred))

Accuracy: 60.5
Precision: 0.6153846153846154
Recall: 0.5279187817258884
F1 Score: 0.5683060109289617
Confusion Matrix:
[[138  65]
 [ 93 104]]
Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.68      0.64       203
           1       0.62      0.53      0.57       197

    accuracy                           0.60       400
   macro avg       0.61      0.60      0.60       400
weighted avg       0.61      0.60      0.60       400



c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [98]:
# create random forest model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
model=RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    criterion='gini',
    min_samples_split=2,
    min_samples_leaf=1,
    )
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
# metrics
accuracy=accuracy_score(y_test,y_pred)
print('Accuracy:', accuracy*100)

# apply hyperparameter tuning using GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_*100)

Accuracy: 71.5
Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation score: 71.75


In [96]:
# check gradient boosting model
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy * 100)

Accuracy: 72.25


In [97]:
# check ada boost model
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy * 100)

Accuracy: 70.0
